## Transformación logaritmica para variables con mucho sesgo

Cargamos el mismo conjunto de datos previo a la estandarización y eliminamos las mismas variables que antes

In [2]:
import pandas as pd
from sklearn.cluster import KMeans
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import unicodedata
import re
import gower

import kmedoids # Nueva librería para PAM
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.spatial.distance as ssd
import prince
import numpy as np

from itertools import combinations
import heapq
from sklearn.preprocessing import StandardScaler

In [3]:
trainset = pd.read_csv("train_imputado.csv")

In [4]:
trainset.head(10)

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,elegibilidad_balance,convulsiones,caidas,fracturas,medicacion_osteoporosis,aprueba_cond4,tamano_manguito_pa,estado_elastografia,tipo_sonda_elasto,raza_etnia
0,46.0,5.0,2.0,4.62,15.0,15.0,30.0,30.0,66.0,111.6,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
1,57.0,3.0,2.0,2.17,15.0,15.0,30.0,30.0,73.0,124.1,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
2,48.0,3.0,2.0,5.00,15.0,15.0,30.0,30.0,67.0,89.3,...,0,0,0,1,0,1,4.0,1,b'M',3.0
3,40.0,5.0,1.0,3.11,15.0,15.0,30.0,30.0,81.0,81.9,...,1,0,0,0,0,1,3.0,1,b'M',2.0
4,66.0,3.0,2.0,3.47,15.0,15.0,30.0,8.0,103.0,77.4,...,1,0,0,0,0,0,3.0,1,b'M',6.0
5,27.0,2.0,1.0,5.00,15.0,15.0,30.0,30.0,69.0,50.8,...,1,0,0,0,0,1,3.0,1,b'M',7.0
6,69.0,1.0,1.0,2.43,15.0,15.0,30.0,30.0,50.0,86.4,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
7,33.0,5.0,2.0,1.05,15.0,15.0,30.0,30.0,58.0,84.2,...,1,0,0,0,0,1,3.0,1,b'M',3.0
8,54.0,4.0,1.0,5.00,15.0,15.0,30.0,22.0,73.0,94.2,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
9,40.0,1.0,1.0,5.00,15.0,15.0,30.0,2.0,51.0,66.5,...,1,0,0,0,0,0,3.0,1,b'M',3.0


In [5]:
# 1. Cargar diccionario



ruta_diccionario = "../data/Diccionario_TFM_CompletoMF.xlsx"
diccionario = pd.read_excel(ruta_diccionario)

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    return x

def limpiar_nombre(col):
    col = col.lower()
    col = col.replace(" ", "_")
    col = re.sub(r'[^a-z0-9_]', '', col)
    return col

diccionario["Tipo de Variable"] = diccionario["Tipo de Variable"].apply(normalizar_texto)

cols_categoricas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("categ", na=False), "Qué es"
].tolist()
var_categoricas = [col for col in cols_categoricas if col in trainset.columns]

cols_numericas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("num", na=False), "Qué es"
].tolist()
var_numericas = [col for col in cols_numericas if col in trainset.columns]

trainset[var_numericas] = trainset[var_numericas].astype('float64')
trainset[var_categoricas] = trainset[var_categoricas].astype('object')

print(f"Número de columnas categóricas: {len(var_categoricas)}")
print(var_categoricas)

print(f"Número de columnas numéricas: {len(var_numericas)}")
print(var_numericas)

Número de columnas categóricas: 16
['genero', 'periodo_examen', 'pais_nacimiento', 'toma_suplementos', 'toma_antiacidos', 'estado_examen_balance', 'elegibilidad_balance', 'convulsiones', 'caidas', 'fracturas', 'medicacion_osteoporosis', 'aprueba_cond4', 'tamano_manguito_pa', 'estado_elastografia', 'tipo_sonda_elasto', 'raza_etnia']
Número de columnas numéricas: 46
['edad_an', 'tamano_hogar', 'psu', 'ratio_pobreza', 'tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3', 'tiempo_seg_cond4', 'pulso', 'peso_kg', 'altura_cm', 'imc', 'largo_pierna_superior_cm', 'largo_brazo_superior_cm', 'medidas_validas_elasto', 'intentos_totales_elasto', 'rigidez_mediana_kpa', 'rigidez_iqr', 'ratio_iqr_mediana', 'cap_mediana_db_m', 'cap_iqr', 'albumina_orina_mg_l', 'creatinina_orina_umol_l', 'ratio_albumina_creatinina', 'peso_flebotomia', 'proteina_c_reactiva_mg_l', 'leucocitos_totales', 'linfocitos_abs', 'monocitos_abs', 'neutrofilos_abs', 'eosinofilos_abs', 'eritrocitos_totales', 'hemoglobina_g_dl',

eliminamos las mismas variables que en el apartado anterior

In [6]:
columnas_a_eliminar = ['tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3', 'peso_kg', 'albumina_orina_mg_l', 'neutrofilos_abs','rigidez_iqr', 'ratio_iqr_mediana','medidas_validas_elasto','intentos_totales_elasto','psu', 'cadmio_sangre_nmol_l', 'mercurio_sangre_nmol_l']

# Utilizamos el método .drop() especificando que queremos eliminar columnas
trainset = trainset.drop(columns=columnas_a_eliminar)
trainset.shape

(1919, 49)

In [7]:
var_num = trainset.select_dtypes(include=['number'])
var_num.shape

(1919, 33)

obtenemos el skew para cada variable: medida estadística que describe la falta de simetría en la distribución de tus datos alrededor de su media.

In [8]:
asimetria = var_num.skew().sort_values(ascending=False)

print("--- Nivel de Asimetría (Skewness) de las Variables ---")
print("Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.\n")
print(asimetria)

--- Nivel de Asimetría (Skewness) de las Variables ---
Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.

plomo_sangre_umol_l               19.482658
ratio_albumina_creatinina         14.215589
rigidez_mediana_kpa                7.974745
proteina_c_reactiva_mg_l           6.952438
trigliceridos_mmol_l               6.916607
eosinofilos_abs                    5.417549
selenio_sangre_umol_l              5.352683
ancho_distribucion_eritrocitos     5.211623
cap_iqr                            1.838814
peso_flebotomia                    1.829170
manganeso_sangre_nmol_l            1.501256
creatinina_orina_umol_l            1.216544
monocitos_abs                      1.209084
linfocitos_abs                     1.156344
imc                                1.138741
leucocitos_totales                 1.137664
hdl_mmol_l                         1.131395
tamano_hogar                       0.884210
plaquetas_totales                  0.787695
volumen_plaquetario_medio       

se puede observar que para ciertas variables la transformación logaritmica no será sido del todo útil esto es debido a:

1. medidas como plomo_sangre_umol_l son muy pequeñas y la la función log1p(x) uma 1 antes de sacar el logaritmo por tanto si se le suma 1 a un valor muy pequeño es practicamnete 1.

Para estos valores se aplica la transformación de raiz cuadrada.

In [9]:
skew_actual = var_num.skew()
var_num_log = var_num.copy()

# 2. Filtrar las variables con asimetría severa
variables_sesgadas = skew_actual[skew_actual.abs() > 1].index.tolist()

print("--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---")
for col in variables_sesgadas:
    print(f"- {col} (Skew original: {skew_actual[col]:.2f})")

# Definimos las variables rebeldes que necesitan raíz cuadrada en lugar de logaritmo
variables_raiz = ['plomo_sangre_umol_l']

print("\n--- APLICANDO TRANSFORMACIONES ---")
# 3. Aplicar la transformación (Logaritmo o Raíz Cuadrada)
for col in variables_sesgadas:
    # Si la variable está en nuestra lista de excepciones, aplicamos raíz cuadrada
    if col in variables_raiz:
        var_num_log[col] = np.sqrt(var_num_log[col])
        print(f"✓ Raíz cuadrada aplicada a: {col}")
    else:
        # Para el resto, verificamos que no haya valores negativos antes de aplicar el logaritmo
        if (var_num_log[col] < 0).any():
            print(f"⚠️ Advertencia: {col} tiene valores negativos. No se aplicó logaritmo.")
        else:
            var_num_log[col] = np.log1p(var_num_log[col])
            # print(f"✓ Logaritmo aplicado a: {col}") # Opcional: descomentar para ver el progreso

print("\nTransformaciones aplicadas con éxito.")

# 4. Verificar los nuevos niveles de asimetría para confirmar la mejora
print("\n--- NUEVA ASIMETRÍA TRAS LAS TRANSFORMACIONES ---")
skew_nuevo = var_num_log[variables_sesgadas].skew()
for col in variables_sesgadas:
    print(f"- {col} (Nuevo Skew: {skew_nuevo[col]:.2f})")

--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---
- imc (Skew original: 1.14)
- rigidez_mediana_kpa (Skew original: 7.97)
- cap_iqr (Skew original: 1.84)
- creatinina_orina_umol_l (Skew original: 1.22)
- ratio_albumina_creatinina (Skew original: 14.22)
- peso_flebotomia (Skew original: 1.83)
- proteina_c_reactiva_mg_l (Skew original: 6.95)
- leucocitos_totales (Skew original: 1.14)
- linfocitos_abs (Skew original: 1.16)
- monocitos_abs (Skew original: 1.21)
- eosinofilos_abs (Skew original: 5.42)
- ancho_distribucion_eritrocitos (Skew original: 5.21)
- trigliceridos_mmol_l (Skew original: 6.92)
- hdl_mmol_l (Skew original: 1.13)
- plomo_sangre_umol_l (Skew original: 19.48)
- selenio_sangre_umol_l (Skew original: 5.35)
- manganeso_sangre_nmol_l (Skew original: 1.50)

--- APLICANDO TRANSFORMACIONES ---
✓ Raíz cuadrada aplicada a: plomo_sangre_umol_l

Transformaciones aplicadas con éxito.

--- NUEVA ASIMETRÍA TRAS LAS TRANSFORMACIONES ---
- imc (Nuevo Skew: 0.42)
- rigidez_mediana_kpa (Nue

In [9]:
print("=" * 80)
print("FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo de variables a agrupar
columnas_totales = var_num_log.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log[list(combo)]
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log[features_a_probar]

            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            labels = kmeans.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'plomo_sangre_umol_l') (Score: 0.7826) | Tamaños N: [1250, 669]
  -> + Variable 3: 'ancho_distribucion_eritrocitos' | Nuevo Score: 0.7816 | Tamaños N: [1250, 669]
  -> + Variable 4: 'selenio_sangre_umol_l' | Nuevo Score: 0.7805 | Tamaños N: [1250, 669]
  -> + Variable 5: 'eosinofilos_abs' | Nuevo Score: 0.7795 | Tamaños N: [1250, 669]
  -> + Variable 6: 'monocitos_abs' | Nuevo Score: 0.7785 | Tamaños N: [1250, 669]
  -> + Variable 7: 'hdl_mmol_l' | Nuevo Score: 0.7770 | Tamaños N: [1250, 669]
  -> + Variable 8: 'linfocitos_abs' | Nuevo Score: 0.7748 | Tamaños N: [1250, 669]
  -> + Variable 9: 'imc' | Nuevo Score: 0.7727 | Tamaños N: [1250, 669]
  -> + Variable 10: 'leucocitos_totales' | Nuevo Score: 0.7704 | Tamaños N: [1250, 669]
  -> + Variable 11: 'trigliceridos_mmol_l' | Nuevo Score: 0.7675 | Tamaños N: [1250, 669]
  -> + Variable 12: 'manganeso_sangre_nmol_l'

Los resultados del anterior clustering aunque prometedores NO SON VALIDOS ya que los datos han de , tener escalas similares. Para ello no se va a realiazr sobre los siguientes grupos:

Grupo 1: Escalas Altas (Medias entre 8 y 35)
Estas variables dominan por completo el cálculo de distancias actualmente. Sus valores suelen oscilar entre 5 y 40.

* conc_hemoglobina_media (Rango: 28.4 - 37.7)

* tiempo_seg_cond4 (Rango: 1.0 - 30.0)

* hemoglobina_g_dl (Rango: 8.1 - 18.6)

* peso_flebotomia (Rango: 8.7 - 12.4)

* volumen_plaquetario_medio (Rango: 5.9 - 12.5)

Grupo 2: Escalas Medias (Medias entre 1 y 5)
Este es el grupo más numeroso. Sus valores están contenidos principalmente entre el 0.5 y el 8.0.

* eritrocitos_totales

* cap_iqr

* ldl_martin_mmol_l

* tamano_hogar

* ancho_distribucion_eritrocitos

* ratio_albumina_creatinina

* rigidez_mediana_kpa

* proteina_c_reactiva_mg_l

* selenio_sangre_umol_l

* linfocitos_abs

Grupo 3: Escalas Bajas o Trazas (Medias menores a 1)
Estas variables son prácticamente invisibles para el algoritmo de clustering si se mezclan con el Grupo 1.

* hdl_mmol_l (Rango: 0.46 - 1.48)

* trigliceridos_mmol_l (Rango: 0.25 - 3.03)

* monocitos_abs (Rango: 0.10 - 0.99)

* eosinofilos_abs (Rango: 0.10 - 1.36)

* plomo_sangre_umol_l (Rango: 0.005 - 1.20)

USANDO K-MEANS

In [10]:
# 1. Definición de los grupos de variables por escala
grupos_escalas = {
    "Grupo 1 (Escalas Altas)": [
        'conc_hemoglobina_media', 'tiempo_seg_cond4', 'hemoglobina_g_dl',
        'peso_flebotomia', 'volumen_plaquetario_medio'
    ],
    "Grupo 2 (Escalas Medias)": [
        'eritrocitos_totales', 'cap_iqr', 'ldl_martin_mmol_l', 'tamano_hogar',
        'ancho_distribucion_eritrocitos', 'ratio_albumina_creatinina',
        'rigidez_mediana_kpa', 'proteina_c_reactiva_mg_l', 'selenio_sangre_umol_l',
        'linfocitos_abs'
    ],
    "Grupo 3 (Escalas Bajas)": [
        'hdl_mmol_l', 'trigliceridos_mmol_l', 'monocitos_abs',
        'eosinofilos_abs', 'plomo_sangre_umol_l'
    ]
}

print("=" * 80)
print("FORWARD SELECTION POR GRUPOS DE ESCALAS")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo general (se ajustará por grupo)

# Iteramos sobre cada grupo de variables
for nombre_grupo, variables_grupo in grupos_escalas.items():
    print("\n" + "=" * 60)
    print(f"🚀 INICIANDO ANÁLISIS PARA: {nombre_grupo}")
    print("=" * 60)

    # Filtrar solo las columnas de var_num_log que existen en el grupo
    columnas_totales_grupo = [col for col in variables_grupo if col in var_num_log.columns]

    # Ajustar el máximo de variables a iterar al tamaño real del grupo
    max_vars_grupo = min(max_variables, len(columnas_totales_grupo))

    if max_vars_grupo < 2:
        print(f"⚠️ El {nombre_grupo} tiene menos de 2 variables disponibles. Saltando...")
        continue

    resultados_finales = {}
    campeon_global_score = -1
    campeon_global_k = -1
    campeon_global_vars = []
    campeon_global_conteos = []

    for k in rango_k:
        print(f"\n[ Iniciando búsqueda para K={k} ]")

        columnas_disponibles = columnas_totales_grupo.copy()
        variables_seleccionadas = []

        # --- PASO 1: Buscar el mejor PAR inicial ---
        mejor_score_par = -1
        mejor_par = ()
        mejor_labels_par = None

        for combo in combinations(columnas_disponibles, 2):
            X_subset = var_num_log[list(combo)]
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            labels = kmeans.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_par:
                mejor_score_par = score
                mejor_par = combo
                mejor_labels_par = labels

        # Consolidar el mejor par
        variables_seleccionadas.extend(mejor_par)
        for var in mejor_par:
            columnas_disponibles.remove(var)

        conteos_par = np.bincount(mejor_labels_par).tolist()

        print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

        historial_scores = [mejor_score_par]
        historial_conteos = [conteos_par]

        # --- PASO 2: Forward Selection ---
        for paso in range(3, max_vars_grupo + 1):
            mejor_score_paso = -1
            mejor_variable_paso = None
            mejor_labels_paso = None

            for var_candidata in columnas_disponibles:
                features_a_probar = variables_seleccionadas + [var_candidata]
                X_subset = var_num_log[features_a_probar]

                kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
                labels = kmeans.fit_predict(X_subset)
                score = silhouette_score(X_subset, labels)

                if score > mejor_score_paso:
                    mejor_score_paso = score
                    mejor_variable_paso = var_candidata
                    mejor_labels_paso = labels

            # Consolidar variable ganadora
            variables_seleccionadas.append(mejor_variable_paso)
            columnas_disponibles.remove(mejor_variable_paso)

            conteos_paso = np.bincount(mejor_labels_paso).tolist()

            historial_scores.append(mejor_score_paso)
            historial_conteos.append(conteos_paso)

            print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

        # --- PASO 3: Encontrar el pico máximo ---
        indice_mejor_momento = historial_scores.index(max(historial_scores))
        score_pico = historial_scores[indice_mejor_momento]
        conteos_pico = historial_conteos[indice_mejor_momento]

        vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

        resultados_finales[k] = {
            'score': score_pico,
            'variables': vars_en_pico,
            'conteos': conteos_pico
        }

        if score_pico > campeon_global_score:
            campeon_global_score = score_pico
            campeon_global_k = k
            campeon_global_vars = vars_en_pico
            campeon_global_conteos = conteos_pico

    # Imprimir resumen final por grupo
    print("\n" + "-" * 60)
    print(f"🏆 RESULTADOS: {nombre_grupo} 🏆")
    print("-" * 60)
    print(f"🌟 EL CAMPEÓN DEL GRUPO:")
    print(f"  -> K Óptimo: {campeon_global_k}")
    print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
    print(f"  -> Distribución (N): {campeon_global_conteos}")
    print(f"  -> Variables ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

FORWARD SELECTION POR GRUPOS DE ESCALAS

🚀 INICIANDO ANÁLISIS PARA: Grupo 1 (Escalas Altas)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7662) | Tamaños N: [1250, 669]
  -> + Variable 3: 'conc_hemoglobina_media' | Nuevo Score: 0.7510 | Tamaños N: [1250, 669]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.7379 | Tamaños N: [1250, 669]
  -> + Variable 5: 'hemoglobina_g_dl' | Nuevo Score: 0.7147 | Tamaños N: [1250, 669]

[ Iniciando búsqueda para K=3 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7641) | Tamaños N: [1076, 473, 370]
  -> + Variable 3: 'conc_hemoglobina_media' | Nuevo Score: 0.7322 | Tamaños N: [1061, 496, 362]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.7035 | Tamaños N: [1060, 496, 363]
  -> + Variable 5: 'hemoglobina_g_dl' | Nuevo Score: 0.6559 | Tamaños N: [1061, 496, 362]

[ Iniciando búsqueda para K=4 ]
  -> Mejor par inicial: ('tiempo_seg_co

Usando Clustering Jerarquico

In [11]:


# 1. Definición de los grupos de variables por escala
grupos_escalas = {
    "Grupo 1 (Escalas Altas)": [
        'conc_hemoglobina_media', 'tiempo_seg_cond4', 'hemoglobina_g_dl',
        'peso_flebotomia', 'volumen_plaquetario_medio'
    ],
    "Grupo 2 (Escalas Medias)": [
        'eritrocitos_totales', 'cap_iqr', 'ldl_martin_mmol_l', 'tamano_hogar',
        'ancho_distribucion_eritrocitos', 'ratio_albumina_creatinina',
        'rigidez_mediana_kpa', 'proteina_c_reactiva_mg_l', 'selenio_sangre_umol_l',
        'linfocitos_abs'
    ],
    "Grupo 3 (Escalas Bajas)": [
        'hdl_mmol_l', 'trigliceridos_mmol_l', 'monocitos_abs',
        'eosinofilos_abs', 'plomo_sangre_umol_l'
    ]
}

print("=" * 80)
print("FORWARD SELECTION POR GRUPOS DE ESCALAS (CLUSTERING JERÁRQUICO)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo general (se ajustará por grupo)

# Iteramos sobre cada grupo de variables
for nombre_grupo, variables_grupo in grupos_escalas.items():
    print("\n" + "=" * 60)
    print(f"🚀 INICIANDO ANÁLISIS PARA: {nombre_grupo}")
    print("=" * 60)

    # Filtrar solo las columnas de var_num_log que existen en el grupo
    columnas_totales_grupo = [col for col in variables_grupo if col in var_num_log.columns]

    # Ajustar el máximo de variables a iterar al tamaño real del grupo
    max_vars_grupo = min(max_variables, len(columnas_totales_grupo))

    if max_vars_grupo < 2:
        print(f"⚠️ El {nombre_grupo} tiene menos de 2 variables disponibles. Saltando...")
        continue

    resultados_finales = {}
    campeon_global_score = -1
    campeon_global_k = -1
    campeon_global_vars = []
    campeon_global_conteos = []

    for k in rango_k:
        print(f"\n[ Iniciando búsqueda para K={k} ]")

        columnas_disponibles = columnas_totales_grupo.copy()
        variables_seleccionadas = []

        # --- PASO 1: Buscar el mejor PAR inicial ---
        mejor_score_par = -1
        mejor_par = ()
        mejor_labels_par = None

        for combo in combinations(columnas_disponibles, 2):
            X_subset = var_num_log[list(combo)]

            # Instanciamos y ajustamos Clustering Jerárquico
            hclust = AgglomerativeClustering(n_clusters=k)
            labels = hclust.fit_predict(X_subset)

            score = silhouette_score(X_subset, labels)

            if score > mejor_score_par:
                mejor_score_par = score
                mejor_par = combo
                mejor_labels_par = labels

        # Consolidar el mejor par
        variables_seleccionadas.extend(mejor_par)
        for var in mejor_par:
            columnas_disponibles.remove(var)

        conteos_par = np.bincount(mejor_labels_par).tolist()

        print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

        historial_scores = [mejor_score_par]
        historial_conteos = [conteos_par]

        # --- PASO 2: Forward Selection ---
        for paso in range(3, max_vars_grupo + 1):
            mejor_score_paso = -1
            mejor_variable_paso = None
            mejor_labels_paso = None

            for var_candidata in columnas_disponibles:
                features_a_probar = variables_seleccionadas + [var_candidata]
                X_subset = var_num_log[features_a_probar]

                # Instanciamos y ajustamos Clustering Jerárquico
                hclust = AgglomerativeClustering(n_clusters=k)
                labels = hclust.fit_predict(X_subset)

                score = silhouette_score(X_subset, labels)

                if score > mejor_score_paso:
                    mejor_score_paso = score
                    mejor_variable_paso = var_candidata
                    mejor_labels_paso = labels

            # Consolidar variable ganadora
            variables_seleccionadas.append(mejor_variable_paso)
            columnas_disponibles.remove(mejor_variable_paso)

            conteos_paso = np.bincount(mejor_labels_paso).tolist()

            historial_scores.append(mejor_score_paso)
            historial_conteos.append(conteos_paso)

            print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

        # --- PASO 3: Encontrar el pico máximo ---
        indice_mejor_momento = historial_scores.index(max(historial_scores))
        score_pico = historial_scores[indice_mejor_momento]
        conteos_pico = historial_conteos[indice_mejor_momento]

        vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

        resultados_finales[k] = {
            'score': score_pico,
            'variables': vars_en_pico,
            'conteos': conteos_pico
        }

        if score_pico > campeon_global_score:
            campeon_global_score = score_pico
            campeon_global_k = k
            campeon_global_vars = vars_en_pico
            campeon_global_conteos = conteos_pico

    # Imprimir resumen final por grupo
    print("\n" + "-" * 60)
    print(f"🏆 RESULTADOS: {nombre_grupo} 🏆")
    print("-" * 60)
    print(f"🌟 EL CAMPEÓN DEL GRUPO:")
    print(f"  -> K Óptimo: {campeon_global_k}")
    print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
    print(f"  -> Distribución (N): {campeon_global_conteos}")
    print(f"  -> Variables ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

FORWARD SELECTION POR GRUPOS DE ESCALAS (CLUSTERING JERÁRQUICO)

🚀 INICIANDO ANÁLISIS PARA: Grupo 1 (Escalas Altas)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7333) | Tamaños N: [1423, 496]
  -> + Variable 3: 'hemoglobina_g_dl' | Nuevo Score: 0.7120 | Tamaños N: [1405, 514]
  -> + Variable 4: 'conc_hemoglobina_media' | Nuevo Score: 0.7031 | Tamaños N: [1406, 513]
  -> + Variable 5: 'volumen_plaquetario_medio' | Nuevo Score: 0.6857 | Tamaños N: [1428, 491]

[ Iniciando búsqueda para K=3 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7692) | Tamaños N: [395, 496, 1028]
  -> + Variable 3: 'volumen_plaquetario_medio' | Nuevo Score: 0.7157 | Tamaños N: [456, 435, 1028]
  -> + Variable 4: 'conc_hemoglobina_media' | Nuevo Score: 0.6951 | Tamaños N: [442, 452, 1025]
  -> + Variable 5: 'hemoglobina_g_dl' | Nuevo Score: 0.6415 | Tamaños N: [417, 491, 1011]

[ Iniciando búsqueda para K=4 ]
  -> Mejor par 

Con Clustering GMM

In [12]:
from sklearn.mixture import GaussianMixture
# 1. Definición de los grupos de variables por escala
grupos_escalas = {
    "Grupo 1 (Escalas Altas)": [
        'conc_hemoglobina_media', 'tiempo_seg_cond4', 'hemoglobina_g_dl',
        'peso_flebotomia', 'volumen_plaquetario_medio'
    ],
    "Grupo 2 (Escalas Medias)": [
        'eritrocitos_totales', 'cap_iqr', 'ldl_martin_mmol_l', 'tamano_hogar',
        'ancho_distribucion_eritrocitos', 'ratio_albumina_creatinina',
        'rigidez_mediana_kpa', 'proteina_c_reactiva_mg_l', 'selenio_sangre_umol_l',
        'linfocitos_abs'
    ],
    "Grupo 3 (Escalas Bajas)": [
        'hdl_mmol_l', 'trigliceridos_mmol_l', 'monocitos_abs',
        'eosinofilos_abs', 'plomo_sangre_umol_l'
    ]
}

print("=" * 80)
print("FORWARD SELECTION POR GRUPOS DE ESCALAS (CLUSTERING GMM)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo general (se ajustará por grupo)

# Iteramos sobre cada grupo de variables
for nombre_grupo, variables_grupo in grupos_escalas.items():
    print("\n" + "=" * 60)
    print(f"🚀 INICIANDO ANÁLISIS PARA: {nombre_grupo}")
    print("=" * 60)

    # Filtrar solo las columnas de var_num_log que existen en el grupo
    columnas_totales_grupo = [col for col in variables_grupo if col in var_num_log.columns]

    # Ajustar el máximo de variables a iterar al tamaño real del grupo
    max_vars_grupo = min(max_variables, len(columnas_totales_grupo))

    if max_vars_grupo < 2:
        print(f"⚠️ El {nombre_grupo} tiene menos de 2 variables disponibles. Saltando...")
        continue

    resultados_finales = {}
    campeon_global_score = -1
    campeon_global_k = -1
    campeon_global_vars = []
    campeon_global_conteos = []

    for k in rango_k:
        print(f"\n[ Iniciando búsqueda para K={k} ]")

        columnas_disponibles = columnas_totales_grupo.copy()
        variables_seleccionadas = []

        # --- PASO 1: Buscar el mejor PAR inicial ---
        mejor_score_par = -1
        mejor_par = ()
        mejor_labels_par = None

        for combo in combinations(columnas_disponibles, 2):
            X_subset = var_num_log[list(combo)]

            # Instanciamos y ajustamos GMM
            gmm = GaussianMixture(n_components=k, random_state=42)
            labels = gmm.fit_predict(X_subset)

            score = silhouette_score(X_subset, labels)

            if score > mejor_score_par:
                mejor_score_par = score
                mejor_par = combo
                mejor_labels_par = labels

        # Consolidar el mejor par
        variables_seleccionadas.extend(mejor_par)
        for var in mejor_par:
            columnas_disponibles.remove(var)

        conteos_par = np.bincount(mejor_labels_par).tolist()

        print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

        historial_scores = [mejor_score_par]
        historial_conteos = [conteos_par]

        # --- PASO 2: Forward Selection ---
        for paso in range(3, max_vars_grupo + 1):
            mejor_score_paso = -1
            mejor_variable_paso = None
            mejor_labels_paso = None

            for var_candidata in columnas_disponibles:
                features_a_probar = variables_seleccionadas + [var_candidata]
                X_subset = var_num_log[features_a_probar]

                # Instanciamos y ajustamos GMM
                gmm = GaussianMixture(n_components=k, random_state=42)
                labels = gmm.fit_predict(X_subset)

                score = silhouette_score(X_subset, labels)

                if score > mejor_score_paso:
                    mejor_score_paso = score
                    mejor_variable_paso = var_candidata
                    mejor_labels_paso = labels

            # Consolidar variable ganadora
            variables_seleccionadas.append(mejor_variable_paso)
            columnas_disponibles.remove(mejor_variable_paso)

            conteos_paso = np.bincount(mejor_labels_paso).tolist()

            historial_scores.append(mejor_score_paso)
            historial_conteos.append(conteos_paso)

            print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

        # --- PASO 3: Encontrar el pico máximo ---
        indice_mejor_momento = historial_scores.index(max(historial_scores))
        score_pico = historial_scores[indice_mejor_momento]
        conteos_pico = historial_conteos[indice_mejor_momento]

        vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

        resultados_finales[k] = {
            'score': score_pico,
            'variables': vars_en_pico,
            'conteos': conteos_pico
        }

        if score_pico > campeon_global_score:
            campeon_global_score = score_pico
            campeon_global_k = k
            campeon_global_vars = vars_en_pico
            campeon_global_conteos = conteos_pico

    # Imprimir resumen final por grupo
    print("\n" + "-" * 60)
    print(f"🏆 RESULTADOS: {nombre_grupo} 🏆")
    print("-" * 60)
    print(f"🌟 EL CAMPEÓN DEL GRUPO:")
    print(f"  -> K Óptimo: {campeon_global_k}")
    print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
    print(f"  -> Distribución (N): {campeon_global_conteos}")
    print(f"  -> Variables ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

FORWARD SELECTION POR GRUPOS DE ESCALAS (CLUSTERING GMM)

🚀 INICIANDO ANÁLISIS PARA: Grupo 1 (Escalas Altas)

[ Iniciando búsqueda para K=2 ]


  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.6667) | Tamaños N: [1006, 913]
  -> + Variable 3: 'conc_hemoglobina_media' | Nuevo Score: 0.6480 | Tamaños N: [1006, 913]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.6326 | Tamaños N: [1006, 913]
  -> + Variable 5: 'hemoglobina_g_dl' | Nuevo Score: 0.6069 | Tamaños N: [1006, 913]

[ Iniciando búsqueda para K=3 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7432) | Tamaños N: [1006, 454, 459]
  -> + Variable 3: 'conc_hemoglobina_media' | Nuevo Score: 0.7106 | Tamaños N: [1006, 460, 453]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.6825 | Tamaños N: [1006, 461, 452]
  -> + Variable 5: 'hemoglobina_g_dl' | Nuevo Score: 0.6354 | Tamaños N: [1006, 460, 453]

[ Iniciando búsqueda para K=4 ]
  -> Mejor par inicial: ('tiempo_seg_cond4', 'peso_flebotomia') (Score: 0.7435) | Tamaños N: [1006, 453, 257, 203]
  -> + Variable 3: 'conc_hemoglobina_media' | Nue

### ESTANDARIZACIÓN
Tras aplicar la transformación logaritmica estandarizamos el conjunto de datos.

In [10]:

scaler = StandardScaler()
datos_escalados_array = scaler.fit_transform(var_num_log)

In [11]:
var_num_log_scaled = pd.DataFrame(
    datos_escalados_array,
    columns=var_num_log.columns, # Recuperamos los nombres de las columnas
    index=var_num_log.index      # Recuperamos los índices originales
)
var_num_log_scaled.head(10)

,edad_an,tamano_hogar,ratio_pobreza,tiempo_seg_cond4,pulso,altura_cm,imc,largo_pierna_superior_cm,largo_brazo_superior_cm,rigidez_mediana_kpa,...,conc_hemoglobina_media,ancho_distribucion_eritrocitos,plaquetas_totales,volumen_plaquetario_medio,trigliceridos_mmol_l,ldl_martin_mmol_l,hdl_mmol_l,plomo_sangre_umol_l,selenio_sangre_umol_l,manganeso_sangre_nmol_l
0,-0.188751,1.575278,0.984181,0.810059,-0.438261,-1.541547,2.237715,-0.965813,-1.984451,0.788129,...,0.434668,0.335281,-0.148788,-0.064880,-0.217923,0.564297,0.991320,-1.128475,0.358438,0.143478
1,0.575265,0.208042,-0.578577,0.810059,0.158633,-0.490585,2.124402,-1.332158,-0.759037,3.035058,...,-3.080375,1.460799,1.324984,-1.395746,2.836066,-0.674377,-1.076357,-0.913991,-0.554631,-2.931749
2,-0.049839,0.208042,1.226568,0.810059,-0.352991,0.621010,0.084882,0.922273,0.641436,-0.790142,...,0.786172,-0.624857,-1.146655,0.933270,-0.629579,0.004270,0.050606,-0.913991,0.328263,1.263090
3,-0.605487,1.575278,0.021012,0.810059,0.840797,1.136385,-0.556256,1.485880,0.816495,-0.969417,...,-0.385509,-0.267257,-0.440472,0.489648,-0.113511,1.921592,0.050606,0.263839,0.298000,-0.321799
4,1.200370,0.208042,0.250642,-1.268443,2.716750,0.378480,-0.420615,0.302304,0.606424,0.820295,...,-0.502677,1.383632,-0.302306,-1.395746,1.260745,-1.470505,0.742922,2.202996,3.024857,0.752001
5,-1.508416,-0.475576,1.226568,0.810059,-0.182450,-1.460704,-1.282279,-1.163075,-1.669345,-0.081482,...,-0.971350,1.227876,-0.102733,1.376892,-1.218732,-1.323370,1.641699,16.169732,0.972513,1.507047
6,1.408738,-1.159194,-0.412733,0.810059,-1.802591,-0.965539,0.777208,-0.655829,-0.513955,1.005687,...,-0.268341,0.335281,-0.271603,0.156931,-0.670505,-0.762202,0.537385,0.318915,-0.488996,0.709928
7,-1.091680,1.575278,-1.292981,0.810059,-1.120426,0.216793,0.024675,0.247010,0.180119,-0.126409,...,-0.151173,-0.444798,0.588098,-0.619407,0.523314,0.291392,-0.394170,0.091336,0.328263,-0.478391
8,0.366897,0.891660,1.226568,0.054240,0.158633,0.348164,0.457969,-0.148582,-0.513955,0.755577,...,1.372012,-0.807507,-0.087381,-0.952124,2.461369,-0.142865,-1.142999,-0.159167,-0.521761,-1.554410
9,-0.605487,-1.159194,1.226568,-1.835307,-1.717320,0.732169,-1.262144,0.386845,0.116258,0.173519,...,0.903340,-0.534508,-0.732157,-0.064880,-1.170429,-1.027958,-0.265184,0.603880,1.224884,-2.172327


In [12]:
var_num_log_scaled.shape


(1919, 33)

K-Means

In [16]:
print("=" * 80)
print("FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            labels = kmeans.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('volumen_corpuscular_medio', 'ancho_distribucion_eritrocitos') (Score: 0.5409) | Tamaños N: [274, 1645]
  -> + Variable 3: 'hemoglobina_g_dl' | Nuevo Score: 0.4607 | Tamaños N: [279, 1640]
  -> + Variable 4: 'plomo_sangre_umol_l' | Nuevo Score: 0.4066 | Tamaños N: [282, 1637]
  -> + Variable 5: 'rigidez_mediana_kpa' | Nuevo Score: 0.3565 | Tamaños N: [279, 1640]
  -> + Variable 6: 'eosinofilos_abs' | Nuevo Score: 0.3254 | Tamaños N: [1656, 263]
  -> + Variable 7: 'ratio_albumina_creatinina' | Nuevo Score: 0.2887 | Tamaños N: [1538, 381]
  -> + Variable 8: 'manganeso_sangre_nmol_l' | Nuevo Score: 0.2672 | Tamaños N: [332, 1587]
  -> + Variable 9: 'trigliceridos_mmol_l' | Nuevo Score: 0.2579 | Tamaños N: [1618, 301]
  -> + Variable 10: 'largo_pierna_superior_cm' | Nuevo Score: 0.2357 | Tamaños N: [1617, 302]
  -> + Variable 11: 'volumen_plaquetario_medio' | Nuevo Score: 0.2214 | Tamaño

Clustering Jerarquico (tras transofrmar y estandarizar)

In [17]:
import numpy as np
from itertools import combinations
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

print("=" * 80)
print("FORWARD SELECTION: CLUSTERING JERÁRQUICO CON TAMAÑO (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]

        # INSTANCIAMOS EL CLUSTERING JERÁRQUICO
        jerarquico = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = jerarquico.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            # INSTANCIAMOS EL CLUSTERING JERÁRQUICO
            jerarquico = AgglomerativeClustering(n_clusters=k, linkage='ward')
            labels = jerarquico.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION: CLUSTERING JERÁRQUICO CON TAMAÑO (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('volumen_corpuscular_medio', 'ancho_distribucion_eritrocitos') (Score: 0.6256) | Tamaños N: [1789, 130]
  -> + Variable 3: 'conc_hemoglobina_media' | Nuevo Score: 0.5999 | Tamaños N: [1829, 90]
  -> + Variable 4: 'hemoglobina_g_dl' | Nuevo Score: 0.5492 | Tamaños N: [1819, 100]
  -> + Variable 5: 'ratio_pobreza' | Nuevo Score: 0.5098 | Tamaños N: [1840, 79]
  -> + Variable 6: 'eritrocitos_totales' | Nuevo Score: 0.4714 | Tamaños N: [1829, 90]
  -> + Variable 7: 'trigliceridos_mmol_l' | Nuevo Score: 0.4283 | Tamaños N: [1822, 97]
  -> + Variable 8: 'ratio_albumina_creatinina' | Nuevo Score: 0.4086 | Tamaños N: [1823, 96]
  -> + Variable 9: 'creatinina_orina_umol_l' | Nuevo Score: 0.3969 | Tamaños N: [1838, 81]
  -> + Variable 10: 'cap_mediana_db_m' | Nuevo Score: 0.3363 | Tamaños N: [1807, 112]
  -> + Variable 11: 'largo_brazo_superior_cm' | Nuevo Score: 0.3547 | Tamaños N: 

Clustering GMM (tras transofrmar y estandarizar)

In [18]:
import numpy as np
from itertools import combinations
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

print("=" * 80)
print("FORWARD SELECTION: GAUSSIAN MIXTURE MODEL (GMM) CON TAMAÑO (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 20    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]

        # INSTANCIAMOS EL GAUSSIAN MIXTURE MODEL
        # n_components es el equivalente a los clústeres en GMM
        gmm = GaussianMixture(n_components=k, random_state=42)
        labels = gmm.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            # INSTANCIAMOS EL GAUSSIAN MIXTURE MODEL
            gmm = GaussianMixture(n_components=k, random_state=42)
            labels = gmm.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION: GAUSSIAN MIXTURE MODEL (GMM) CON TAMAÑO (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('plomo_sangre_umol_l', 'selenio_sangre_umol_l') (Score: 0.5880) | Tamaños N: [1794, 125]
  -> + Variable 3: 'tamano_hogar' | Nuevo Score: 0.4863 | Tamaños N: [1797, 122]
  -> + Variable 4: 'cap_iqr' | Nuevo Score: 0.4202 | Tamaños N: [1791, 128]
  -> + Variable 5: 'eritrocitos_totales' | Nuevo Score: 0.3725 | Tamaños N: [136, 1783]
  -> + Variable 6: 'volumen_plaquetario_medio' | Nuevo Score: 0.3407 | Tamaños N: [142, 1777]
  -> + Variable 7: 'ldl_martin_mmol_l' | Nuevo Score: 0.3118 | Tamaños N: [140, 1779]
  -> + Variable 8: 'ratio_pobreza' | Nuevo Score: 0.2965 | Tamaños N: [1801, 118]
  -> + Variable 9: 'cap_mediana_db_m' | Nuevo Score: 0.2554 | Tamaños N: [157, 1762]
  -> + Variable 10: 'plaquetas_totales' | Nuevo Score: 0.2397 | Tamaños N: [1767, 152]
  -> + Variable 11: 'proteina_c_reactiva_mg_l' | Nuevo Score: 0.2224 | Tamaños N: [1732, 187]
  -> + Variable 1

In [15]:
#comando de LIMPIEZA DE KERNEL PARA EVITAR QUE SE MEZCLEN VARIBALES DE DISTINTAS PRUEBAS
%reset -f
%reset -f in out dhist

Flushing input history
Flushing output cache (0 entries)
Flushing directory history


------------------------------------------------------------------------------------
------------------------------------------------------------------------------------
------------------------------------------------------------------------------------


### Clustering con variables del codigo genético.

In [1]:
import pandas as pd
from sklearn.cluster import KMeans
from kmodes.kprototypes import KPrototypes
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import unicodedata
import re
import gower

import kmedoids # Nueva librería para PAM
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

from scipy.cluster.hierarchy import dendrogram, linkage
import scipy.spatial.distance as ssd
import prince
import numpy as np

from itertools import combinations
import heapq
from sklearn.preprocessing import StandardScaler

In [2]:
trainset = pd.read_csv("train_imputado.csv")

In [3]:
trainset.head(10)

,edad_an,tamano_hogar,psu,ratio_pobreza,tiempo_seg_cond1,tiempo_seg_cond2,tiempo_seg_cond3,tiempo_seg_cond4,pulso,peso_kg,...,elegibilidad_balance,convulsiones,caidas,fracturas,medicacion_osteoporosis,aprueba_cond4,tamano_manguito_pa,estado_elastografia,tipo_sonda_elasto,raza_etnia
0,46.0,5.0,2.0,4.62,15.0,15.0,30.0,30.0,66.0,111.6,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
1,57.0,3.0,2.0,2.17,15.0,15.0,30.0,30.0,73.0,124.1,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
2,48.0,3.0,2.0,5.00,15.0,15.0,30.0,30.0,67.0,89.3,...,0,0,0,1,0,1,4.0,1,b'M',3.0
3,40.0,5.0,1.0,3.11,15.0,15.0,30.0,30.0,81.0,81.9,...,1,0,0,0,0,1,3.0,1,b'M',2.0
4,66.0,3.0,2.0,3.47,15.0,15.0,30.0,8.0,103.0,77.4,...,1,0,0,0,0,0,3.0,1,b'M',6.0
5,27.0,2.0,1.0,5.00,15.0,15.0,30.0,30.0,69.0,50.8,...,1,0,0,0,0,1,3.0,1,b'M',7.0
6,69.0,1.0,1.0,2.43,15.0,15.0,30.0,30.0,50.0,86.4,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
7,33.0,5.0,2.0,1.05,15.0,15.0,30.0,30.0,58.0,84.2,...,1,0,0,0,0,1,3.0,1,b'M',3.0
8,54.0,4.0,1.0,5.00,15.0,15.0,30.0,22.0,73.0,94.2,...,1,0,0,0,0,1,4.0,1,b'XL',3.0
9,40.0,1.0,1.0,5.00,15.0,15.0,30.0,2.0,51.0,66.5,...,1,0,0,0,0,0,3.0,1,b'M',3.0


In [4]:
# 1. Cargar diccionario



ruta_diccionario = "../data/Diccionario_TFM_CompletoMF.xlsx"
diccionario = pd.read_excel(ruta_diccionario)

def normalizar_texto(x):
    x = str(x).strip().lower()
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode("utf-8")
    return x

def limpiar_nombre(col):
    col = col.lower()
    col = col.replace(" ", "_")
    col = re.sub(r'[^a-z0-9_]', '', col)
    return col

diccionario["Tipo de Variable"] = diccionario["Tipo de Variable"].apply(normalizar_texto)

cols_categoricas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("categ", na=False), "Qué es"
].tolist()
var_categoricas = [col for col in cols_categoricas if col in trainset.columns]

cols_numericas = diccionario.loc[
    diccionario["Tipo de Variable"].str.contains("num", na=False), "Qué es"
].tolist()
var_numericas = [col for col in cols_numericas if col in trainset.columns]

trainset[var_numericas] = trainset[var_numericas].astype('float64')
trainset[var_categoricas] = trainset[var_categoricas].astype('object')

print(f"Número de columnas categóricas: {len(var_categoricas)}")
print(var_categoricas)

print(f"Número de columnas numéricas: {len(var_numericas)}")
print(var_numericas)

Número de columnas categóricas: 16
['genero', 'periodo_examen', 'pais_nacimiento', 'toma_suplementos', 'toma_antiacidos', 'estado_examen_balance', 'elegibilidad_balance', 'convulsiones', 'caidas', 'fracturas', 'medicacion_osteoporosis', 'aprueba_cond4', 'tamano_manguito_pa', 'estado_elastografia', 'tipo_sonda_elasto', 'raza_etnia']
Número de columnas numéricas: 46
['edad_an', 'tamano_hogar', 'psu', 'ratio_pobreza', 'tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3', 'tiempo_seg_cond4', 'pulso', 'peso_kg', 'altura_cm', 'imc', 'largo_pierna_superior_cm', 'largo_brazo_superior_cm', 'medidas_validas_elasto', 'intentos_totales_elasto', 'rigidez_mediana_kpa', 'rigidez_iqr', 'ratio_iqr_mediana', 'cap_mediana_db_m', 'cap_iqr', 'albumina_orina_mg_l', 'creatinina_orina_umol_l', 'ratio_albumina_creatinina', 'peso_flebotomia', 'proteina_c_reactiva_mg_l', 'leucocitos_totales', 'linfocitos_abs', 'monocitos_abs', 'neutrofilos_abs', 'eosinofilos_abs', 'eritrocitos_totales', 'hemoglobina_g_dl',

eliminamos las mismas variables que en el apartado anterior

In [5]:
columnas_a_eliminar = ['tiempo_seg_cond1', 'tiempo_seg_cond2', 'tiempo_seg_cond3',  'albumina_orina_mg_l', 'neutrofilos_abs','rigidez_iqr', 'ratio_iqr_mediana','medidas_validas_elasto','intentos_totales_elasto','psu']

# Utilizamos el método .drop() especificando que queremos eliminar columnas
trainset = trainset.drop(columns=columnas_a_eliminar)
trainset.shape

(1919, 52)

In [6]:
var_num = trainset.select_dtypes(include=['number'])
var_num.shape

(1919, 36)

obtenemos el skew para cada variable: medida estadística que describe la falta de simetría en la distribución de tus datos alrededor de su media.

In [7]:
asimetria = var_num.skew().sort_values(ascending=False)

print("--- Nivel de Asimetría (Skewness) de las Variables ---")
print("Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.\n")
print(asimetria)

--- Nivel de Asimetría (Skewness) de las Variables ---
Valores ideales: entre -1 y 1. Valores > 1 o < -1 indican asimetría severa.

plomo_sangre_umol_l               19.482658
ratio_albumina_creatinina         14.215589
rigidez_mediana_kpa                7.974745
proteina_c_reactiva_mg_l           6.952438
trigliceridos_mmol_l               6.916607
cadmio_sangre_nmol_l               5.434901
eosinofilos_abs                    5.417549
selenio_sangre_umol_l              5.352683
ancho_distribucion_eritrocitos     5.211623
mercurio_sangre_nmol_l             4.743745
cap_iqr                            1.838814
peso_flebotomia                    1.829170
manganeso_sangre_nmol_l            1.501256
creatinina_orina_umol_l            1.216544
monocitos_abs                      1.209084
linfocitos_abs                     1.156344
imc                                1.138741
leucocitos_totales                 1.137664
hdl_mmol_l                         1.131395
peso_kg                         

se puede observar que para ciertas variables la transformación logaritmica no será sido del todo útil esto es debido a:

1. medidas como plomo_sangre_umol_l son muy pequeñas y la la función log1p(x) uma 1 antes de sacar el logaritmo por tanto si se le suma 1 a un valor muy pequeño es practicamnete 1.

Para estos valores se aplica la transformación de raiz cuadrada.

In [8]:
skew_actual = var_num.skew()
var_num_log = var_num.copy()

# 2. Filtrar las variables con asimetría severa
variables_sesgadas = skew_actual[skew_actual.abs() > 1].index.tolist()

print("--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---")
for col in variables_sesgadas:
    print(f"- {col} (Skew original: {skew_actual[col]:.2f})")

# Definimos las variables rebeldes que necesitan raíz cuadrada en lugar de logaritmo
variables_raiz = ['plomo_sangre_umol_l']

print("\n--- APLICANDO TRANSFORMACIONES ---")
# 3. Aplicar la transformación (Logaritmo o Raíz Cuadrada)
for col in variables_sesgadas:
    # Si la variable está en nuestra lista de excepciones, aplicamos raíz cuadrada
    if col in variables_raiz:
        var_num_log[col] = np.sqrt(var_num_log[col])
        print(f"✓ Raíz cuadrada aplicada a: {col}")
    else:
        # Para el resto, verificamos que no haya valores negativos antes de aplicar el logaritmo
        if (var_num_log[col] < 0).any():
            print(f"⚠️ Advertencia: {col} tiene valores negativos. No se aplicó logaritmo.")
        else:
            var_num_log[col] = np.log1p(var_num_log[col])
            # print(f"✓ Logaritmo aplicado a: {col}") # Opcional: descomentar para ver el progreso

print("\nTransformaciones aplicadas con éxito.")

# 4. Verificar los nuevos niveles de asimetría para confirmar la mejora
print("\n--- NUEVA ASIMETRÍA TRAS LAS TRANSFORMACIONES ---")
skew_nuevo = var_num_log[variables_sesgadas].skew()
for col in variables_sesgadas:
    print(f"- {col} (Nuevo Skew: {skew_nuevo[col]:.2f})")

--- VARIABLES DETECTADAS PARA TRANSFORMACIÓN ---
- imc (Skew original: 1.14)
- rigidez_mediana_kpa (Skew original: 7.97)
- cap_iqr (Skew original: 1.84)
- creatinina_orina_umol_l (Skew original: 1.22)
- ratio_albumina_creatinina (Skew original: 14.22)
- peso_flebotomia (Skew original: 1.83)
- proteina_c_reactiva_mg_l (Skew original: 6.95)
- leucocitos_totales (Skew original: 1.14)
- linfocitos_abs (Skew original: 1.16)
- monocitos_abs (Skew original: 1.21)
- eosinofilos_abs (Skew original: 5.42)
- ancho_distribucion_eritrocitos (Skew original: 5.21)
- trigliceridos_mmol_l (Skew original: 6.92)
- hdl_mmol_l (Skew original: 1.13)
- plomo_sangre_umol_l (Skew original: 19.48)
- cadmio_sangre_nmol_l (Skew original: 5.43)
- mercurio_sangre_nmol_l (Skew original: 4.74)
- selenio_sangre_umol_l (Skew original: 5.35)
- manganeso_sangre_nmol_l (Skew original: 1.50)

--- APLICANDO TRANSFORMACIONES ---
✓ Raíz cuadrada aplicada a: plomo_sangre_umol_l

Transformaciones aplicadas con éxito.

--- NUEVA

In [9]:

scaler = StandardScaler()
datos_escalados_array = scaler.fit_transform(var_num_log)

In [10]:
var_num_log_scaled = pd.DataFrame(
    datos_escalados_array,
    columns=var_num_log.columns, # Recuperamos los nombres de las columnas
    index=var_num_log.index      # Recuperamos los índices originales
)
var_num_log_scaled.head(10)

,edad_an,tamano_hogar,ratio_pobreza,tiempo_seg_cond4,pulso,peso_kg,altura_cm,imc,largo_pierna_superior_cm,largo_brazo_superior_cm,...,plaquetas_totales,volumen_plaquetario_medio,trigliceridos_mmol_l,ldl_martin_mmol_l,hdl_mmol_l,plomo_sangre_umol_l,cadmio_sangre_nmol_l,mercurio_sangre_nmol_l,selenio_sangre_umol_l,manganeso_sangre_nmol_l
0,-0.188751,1.575278,0.984181,0.810059,-0.438261,1.282766,-1.541547,2.237715,-0.965813,-1.984451,...,-0.148788,-0.064880,-0.217923,0.564297,0.991320,-1.128475,-0.198237,0.910452,0.358438,0.143478
1,0.575265,0.208042,-0.578577,0.810059,0.158633,1.859173,-0.490585,2.124402,-1.332158,-0.759037,...,1.324984,-1.395746,2.836066,-0.674377,-1.076357,-0.913991,0.978343,-1.202643,-0.554631,-2.931749
2,-0.049839,0.208042,1.226568,0.810059,-0.352991,0.254454,0.621010,0.084882,0.922273,0.641436,...,-1.146655,0.933270,-0.629579,0.004270,0.050606,-0.913991,0.038614,-0.144919,0.328263,1.263090
3,-0.605487,1.575278,0.021012,0.810059,0.840797,-0.086779,1.136385,-0.556256,1.485880,0.816495,...,-0.440472,0.489648,-0.113511,1.921592,0.050606,0.263839,-0.290164,1.259538,0.298000,-0.321799
4,1.200370,0.208042,0.250642,-1.268443,2.716750,-0.294286,0.378480,-0.420615,0.302304,0.606424,...,-0.302306,-1.395746,1.260745,-1.470505,0.742922,2.202996,3.421947,3.363510,3.024857,0.752001
5,-1.508416,-0.475576,1.226568,0.810059,-0.182450,-1.520881,-1.460704,-1.282279,-1.163075,-1.669345,...,-0.102733,1.376892,-1.218732,-1.323370,1.641699,16.169732,0.345159,-0.976735,0.972513,1.507047
6,1.408738,-1.159194,-0.412733,0.810059,-1.802591,0.120728,-0.965539,0.777208,-0.655829,-0.513955,...,-0.271603,0.156931,-0.670505,-0.762202,0.537385,0.318915,0.906603,-0.011673,-0.488996,0.709928
7,-1.091680,1.575278,-1.292981,0.810059,-1.120426,0.019280,0.216793,0.024675,0.247010,0.180119,...,0.588098,-0.619407,0.523314,0.291392,-0.394170,0.091336,-0.454188,-0.947824,0.328263,-0.478391
8,0.366897,0.891660,1.226568,0.054240,0.158633,0.480406,0.348164,0.457969,-0.148582,-0.513955,...,-0.087381,-0.952124,2.461369,-0.142865,-1.142999,-0.159167,-0.518173,-0.024319,-0.521761,-1.554410
9,-0.605487,-1.159194,1.226568,-1.835307,-1.717320,-0.796913,0.732169,-1.262144,0.386845,0.116258,...,-0.732157,-0.064880,-1.170429,-1.027958,-0.265184,0.603880,-0.814587,1.384638,1.224884,-2.172327


In [11]:
var_num_log_scaled.shape


(1919, 36)

In [12]:


columnas_genetico_numericas = [
    "largo_brazo_superior_cm",
    "mercurio_sangre_nmol_l",
    "hdl_mmol_l",
    "creatinina_orina_umol_l",
    "hemoglobina_g_dl",
    "peso_kg",
    "volumen_plaquetario_medio",
    "ancho_distribucion_eritrocitos",
    "conc_hemoglobina_media",
    "tamano_hogar",
    "pulso",
    "plaquetas_totales",
    "edad_an",
    "trigliceridos_mmol_l"
]

Var_num_log_scaled_original = var_num_log_scaled.copy()
var_num_log_scaled = var_num_log_scaled[columnas_genetico_numericas]
var_num_log_scaled.shape  

(1919, 14)

In [13]:
var_num_log_scaled.head()

,largo_brazo_superior_cm,mercurio_sangre_nmol_l,hdl_mmol_l,creatinina_orina_umol_l,hemoglobina_g_dl,peso_kg,volumen_plaquetario_medio,ancho_distribucion_eritrocitos,conc_hemoglobina_media,tamano_hogar,pulso,plaquetas_totales,edad_an,trigliceridos_mmol_l
0,-1.984451,0.910452,0.991320,0.853000,-0.687034,1.282766,-0.064880,0.335281,0.434668,1.575278,-0.438261,-0.148788,-0.188751,-0.217923
1,-0.759037,-1.202643,-1.076357,-0.631893,-1.107096,1.859173,-1.395746,1.460799,-3.080375,0.208042,0.158633,1.324984,0.575265,2.836066
2,0.641436,-0.144919,0.050606,1.006903,-0.196961,0.254454,0.933270,-0.624857,0.786172,0.208042,-0.352991,-1.146655,-0.049839,-0.629579
3,0.816495,1.259538,0.050606,1.074952,0.433133,-0.086779,0.489648,-0.267257,-0.385509,1.575278,0.840797,-0.440472,-0.605487,-0.113511
4,0.606424,3.363510,0.742922,0.125142,-0.126951,-0.294286,-1.395746,1.383632,-0.502677,0.208042,2.716750,-0.302306,1.200370,1.260745


K-Means

In [14]:
print("=" * 80)
print("FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 14    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]
        kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
        labels = kmeans.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
            labels = kmeans.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION CON TAMAÑO DE CLÚSTERES (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('hemoglobina_g_dl', 'ancho_distribucion_eritrocitos') (Score: 0.4765) | Tamaños N: [1590, 329]
  -> + Variable 3: 'pulso' | Nuevo Score: 0.3565 | Tamaños N: [1589, 330]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.2931 | Tamaños N: [1598, 321]
  -> + Variable 5: 'peso_kg' | Nuevo Score: 0.2521 | Tamaños N: [1509, 410]
  -> + Variable 6: 'tamano_hogar' | Nuevo Score: 0.2192 | Tamaños N: [1518, 401]
  -> + Variable 7: 'conc_hemoglobina_media' | Nuevo Score: 0.1897 | Tamaños N: [1391, 528]
  -> + Variable 8: 'mercurio_sangre_nmol_l' | Nuevo Score: 0.1670 | Tamaños N: [1385, 534]
  -> + Variable 9: 'plaquetas_totales' | Nuevo Score: 0.1552 | Tamaños N: [559, 1360]
  -> + Variable 10: 'edad_an' | Nuevo Score: 0.1344 | Tamaños N: [1329, 590]
  -> + Variable 11: 'hdl_mmol_l' | Nuevo Score: 0.1144 | Tamaños N: [1230, 689]
  -> + Variable 12: 'largo_brazo_superior_cm' | Nue

Clustering Jerarquico (tras transofrmar y estandarizar)

In [15]:
import numpy as np
from itertools import combinations
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

print("=" * 80)
print("FORWARD SELECTION: CLUSTERING JERÁRQUICO CON TAMAÑO (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 14    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]

        # INSTANCIAMOS EL CLUSTERING JERÁRQUICO
        jerarquico = AgglomerativeClustering(n_clusters=k, linkage='ward')
        labels = jerarquico.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            # INSTANCIAMOS EL CLUSTERING JERÁRQUICO
            jerarquico = AgglomerativeClustering(n_clusters=k, linkage='ward')
            labels = jerarquico.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION: CLUSTERING JERÁRQUICO CON TAMAÑO (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('ancho_distribucion_eritrocitos', 'trigliceridos_mmol_l') (Score: 0.5572) | Tamaños N: [1821, 98]
  -> + Variable 3: 'hemoglobina_g_dl' | Nuevo Score: 0.3603 | Tamaños N: [1650, 269]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.4333 | Tamaños N: [1820, 99]
  -> + Variable 5: 'conc_hemoglobina_media' | Nuevo Score: 0.3943 | Tamaños N: [1804, 115]
  -> + Variable 6: 'pulso' | Nuevo Score: 0.3846 | Tamaños N: [1832, 87]
  -> + Variable 7: 'creatinina_orina_umol_l' | Nuevo Score: 0.3562 | Tamaños N: [1825, 94]
  -> + Variable 8: 'peso_kg' | Nuevo Score: 0.3029 | Tamaños N: [1801, 118]
  -> + Variable 9: 'tamano_hogar' | Nuevo Score: 0.2435 | Tamaños N: [1782, 137]
  -> + Variable 10: 'edad_an' | Nuevo Score: 0.2885 | Tamaños N: [1835, 84]
  -> + Variable 11: 'plaquetas_totales' | Nuevo Score: 0.2099 | Tamaños N: [1778, 141]
  -> + Variable 12: 'mercurio_sangr

Clustering GMM (tras transofrmar y estandarizar)

In [16]:
import numpy as np
from itertools import combinations
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

print("=" * 80)
print("FORWARD SELECTION: GAUSSIAN MIXTURE MODEL (GMM) CON TAMAÑO (N)")
print("=" * 80)

rango_k = range(2, 6) # K=2, 3, 4, 5
max_variables = 14    # Límite máximo de variables a agrupar
columnas_totales = var_num_log_scaled.columns.tolist()

resultados_finales = {}
campeon_global_score = -1
campeon_global_k = -1
campeon_global_vars = []
campeon_global_conteos = []

for k in rango_k:
    print(f"\n[ Iniciando búsqueda para K={k} ]")

    columnas_disponibles = columnas_totales.copy()
    variables_seleccionadas = []

    # --- PASO 1: Buscar el mejor PAR inicial (La base sólida) ---
    mejor_score_par = -1
    mejor_par = ()
    mejor_labels_par = None

    # Probamos las parejas posibles
    for combo in combinations(columnas_disponibles, 2):
        X_subset = var_num_log_scaled[list(combo)]

        # INSTANCIAMOS EL GAUSSIAN MIXTURE MODEL
        # n_components es el equivalente a los clústeres en GMM
        gmm = GaussianMixture(n_components=k, random_state=42)
        labels = gmm.fit_predict(X_subset)
        score = silhouette_score(X_subset, labels)

        if score > mejor_score_par:
            mejor_score_par = score
            mejor_par = combo
            mejor_labels_par = labels # Guardamos las etiquetas ganadoras

    # Fijamos el mejor par y lo sacamos de las disponibles
    variables_seleccionadas.extend(mejor_par)
    for var in mejor_par:
        columnas_disponibles.remove(var)

    # Calculamos N por clúster
    conteos_par = np.bincount(mejor_labels_par).tolist()

    print(f"  -> Mejor par inicial: {mejor_par} (Score: {mejor_score_par:.4f}) | Tamaños N: {conteos_par}")

    # Guardamos el historial de scores y conteos
    historial_scores = [mejor_score_par]
    historial_conteos = [conteos_par]

    # --- PASO 2: Forward Selection (Añadir 1 a 1 hasta max_variables) ---
    for paso in range(3, max_variables + 1):
        mejor_score_paso = -1
        mejor_variable_paso = None
        mejor_labels_paso = None

        # Probamos a añadir cada una de las variables que quedan libres
        for var_candidata in columnas_disponibles:
            features_a_probar = variables_seleccionadas + [var_candidata]
            X_subset = var_num_log_scaled[features_a_probar]

            # INSTANCIAMOS EL GAUSSIAN MIXTURE MODEL
            gmm = GaussianMixture(n_components=k, random_state=42)
            labels = gmm.fit_predict(X_subset)
            score = silhouette_score(X_subset, labels)

            if score > mejor_score_paso:
                mejor_score_paso = score
                mejor_variable_paso = var_candidata
                mejor_labels_paso = labels # Guardamos las etiquetas ganadoras

        # Consolidamos la variable ganadora de este paso
        variables_seleccionadas.append(mejor_variable_paso)
        columnas_disponibles.remove(mejor_variable_paso)

        # Calculamos N por clúster
        conteos_paso = np.bincount(mejor_labels_paso).tolist()

        historial_scores.append(mejor_score_paso)
        historial_conteos.append(conteos_paso)

        print(f"  -> + Variable {paso}: '{mejor_variable_paso}' | Nuevo Score: {mejor_score_paso:.4f} | Tamaños N: {conteos_paso}")

    # --- PASO 3: Encontrar el pico máximo para este K ---
    indice_mejor_momento = historial_scores.index(max(historial_scores))
    score_pico = historial_scores[indice_mejor_momento]
    conteos_pico = historial_conteos[indice_mejor_momento]

    # El índice 0 equivale a 2 variables, índice 1 a 3 variables, etc.
    vars_en_pico = variables_seleccionadas[:indice_mejor_momento + 2]

    resultados_finales[k] = {
        'score': score_pico,
        'variables': vars_en_pico,
        'conteos': conteos_pico
    }

    # Actualizar campeón global absoluto
    if score_pico > campeon_global_score:
        campeon_global_score = score_pico
        campeon_global_k = k
        campeon_global_vars = vars_en_pico
        campeon_global_conteos = conteos_pico

print("\n" + "=" * 80)
print("🏆 RESULTADOS FINALES: FORWARD SELECTION 🏆")
print("=" * 80)

print(f"\n🌟 EL CAMPEÓN ABSOLUTO:")
print(f"  -> K Óptimo: {campeon_global_k}")
print(f"  -> Silhouette Score Máximo: {campeon_global_score:.4f}")
print(f"  -> Distribución de Pacientes (N): {campeon_global_conteos}")
print(f"  -> Variables óptimas ({len(campeon_global_vars)}): {', '.join(campeon_global_vars)}")

print("\n--- MEJOR COMBINACIÓN ENCONTRADA PARA CADA K ---")
for k in rango_k:
    res = resultados_finales[k]
    print(f"\nPara K={k}:")
    print(f"  -> Silhouette Score: {res['score']:.4f}")
    print(f"  -> Distribución de Pacientes (N): {res['conteos']}")
    print(f"  -> N Variables: {len(res['variables'])}")
    print(f"  -> Variables: {', '.join(res['variables'])}")

FORWARD SELECTION: GAUSSIAN MIXTURE MODEL (GMM) CON TAMAÑO (N)

[ Iniciando búsqueda para K=2 ]
  -> Mejor par inicial: ('ancho_distribucion_eritrocitos', 'conc_hemoglobina_media') (Score: 0.5243) | Tamaños N: [1682, 237]
  -> + Variable 3: 'hemoglobina_g_dl' | Nuevo Score: 0.4449 | Tamaños N: [1668, 251]
  -> + Variable 4: 'volumen_plaquetario_medio' | Nuevo Score: 0.3799 | Tamaños N: [1655, 264]
  -> + Variable 5: 'pulso' | Nuevo Score: 0.3260 | Tamaños N: [1654, 265]
  -> + Variable 6: 'creatinina_orina_umol_l' | Nuevo Score: 0.2845 | Tamaños N: [1655, 264]
  -> + Variable 7: 'tamano_hogar' | Nuevo Score: 0.2498 | Tamaños N: [1650, 269]
  -> + Variable 8: 'plaquetas_totales' | Nuevo Score: 0.2243 | Tamaños N: [1615, 304]
  -> + Variable 9: 'largo_brazo_superior_cm' | Nuevo Score: 0.1991 | Tamaños N: [1589, 330]
  -> + Variable 10: 'peso_kg' | Nuevo Score: 0.1869 | Tamaños N: [1546, 373]
  -> + Variable 11: 'hdl_mmol_l' | Nuevo Score: 0.1743 | Tamaños N: [1552, 367]
  -> + Variable 1